# 🔬 Lab Module 1 — Xây Dựng LoanBot Phiên Bản Đầu Tiên

> **Chương trình:** AI Agent Harness Engineering  
> **Module:** 1 — Nền Tảng  
> **Thời gian:** ~60 phút  
> **Mục tiêu:** Xây dựng một AI agent thẩm định hồ sơ vay đơn giản cho FinTech Corp

---

## 🏢 Câu chuyện: FinTech Corp cần LoanBot v0.1

FinTech Corp muốn có một prototype đầu tiên của LoanBot để demo cho ban giám đốc.  
Prototype này không cần kết nối hệ thống thực — chúng ta sẽ dùng **mock functions** để mô phỏng.

**Mục tiêu của lab này:**
1. ✅ Hiểu cấu trúc Tool Definition
2. ✅ Xây dựng 5 mock tools cho LoanBot
3. ✅ Kết nối với Anthropic Claude API
4. ✅ Chạy Agent Loop đầu tiên
5. ✅ Quan sát LoanBot xử lý một hồ sơ vay thực tế

---

## ⚙️ Bước 0: Cài đặt môi trường

In [ ]:
# Cài thư viện cần thiết
# Chạy cell này một lần duy nhất
import subprocess
subprocess.run(['pip', 'install', 'anthropic>=0.40.0', '--quiet'], check=True)
print('✅ Cài đặt hoàn tất!')

In [ ]:
import anthropic
import json
import os
from datetime import datetime
from typing import Any

print('✅ Import thành công!')
print(f'📦 Anthropic SDK version: {anthropic.__version__}')

### 🔑 Cấu hình API Key

Bạn cần API key từ [console.anthropic.com](https://console.anthropic.com).  
Thay `'your-api-key-here'` bằng key thực của bạn.

> **Bảo mật:** Không bao giờ commit API key vào git. Trong production, dùng environment variable hoặc secret manager.

In [ ]:
# Cách 1: Nhập trực tiếp (chỉ dùng để học, không dùng trong production)
API_KEY = os.environ.get('ANTHROPIC_API_KEY', 'your-api-key-here')

if API_KEY == 'your-api-key-here':
    print('⚠️  Chưa có API key. Hãy thay bằng key thực của bạn.')
    print('   Lấy key tại: https://console.anthropic.com')
else:
    print(f'✅ API key đã được cấu hình (***{API_KEY[-4:]})')

client = anthropic.Anthropic(api_key=API_KEY)

---

## 🔧 Bước 1: Định Nghĩa 5 Tools Của LoanBot

Trong bài giảng, chúng ta đã học:
- Tool là **function** mà agent có thể gọi để tương tác với thế giới thực
- Mỗi tool có **định nghĩa** (name, description, parameters) để LLM biết khi nào và cách gọi

Dưới đây chúng ta định nghĩa 5 tools của LoanBot theo chuẩn Anthropic SDK:

In [ ]:
# ═══════════════════════════════════════════════════════
# BƯỚC 1A: ĐỊNH NGHĨA TOOLS (Tool Schemas)
# LLM đọc phần này để biết tool nào có sẵn và cách dùng
# ═══════════════════════════════════════════════════════

LOANBOT_TOOLS = [
    {
        "name": "check_credit_score",
        "description": (
            "Truy vấn điểm tín dụng (credit score) của khách hàng từ hệ thống CIC. "
            "Gọi tool này ĐẦU TIÊN khi thẩm định bất kỳ hồ sơ vay nào. "
            "Score 750+ = xuất sắc, 700-749 = tốt, 650-699 = trung bình, dưới 650 = kém."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "customer_id": {
                    "type": "string",
                    "description": "Mã khách hàng nội bộ (ví dụ: 'FC-2024-001')"
                },
                "include_history": {
                    "type": "boolean",
                    "description": "Có bao gồm lịch sử thanh toán 24 tháng không",
                    "default": True
                }
            },
            "required": ["customer_id"]
        }
    },
    {
        "name": "verify_income",
        "description": (
            "Xác minh thu nhập thực tế của khách hàng qua BHXH và sao kê ngân hàng. "
            "So sánh với thu nhập khai báo. Gọi sau check_credit_score."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "customer_id": {"type": "string", "description": "Mã khách hàng"},
                "declared_income": {
                    "type": "number",
                    "description": "Thu nhập khách hàng khai báo (triệu VNĐ/tháng)"
                }
            },
            "required": ["customer_id", "declared_income"]
        }
    },
    {
        "name": "check_blacklist",
        "description": (
            "Kiểm tra khách hàng có trong danh sách nợ xấu NHNN hoặc blacklist nội bộ không. "
            "Nếu is_blacklisted=True, DỪNG ngay và từ chối hồ sơ."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "customer_id": {"type": "string"},
                "national_id": {
                    "type": "string",
                    "description": "Số CCCD/CMND của khách hàng"
                }
            },
            "required": ["customer_id", "national_id"]
        }
    },
    {
        "name": "calculate_dti",
        "description": (
            "Tính Debt-to-Income Ratio (DTI) — tỷ lệ nợ trên thu nhập hàng tháng. "
            "FinTech Corp chỉ chấp nhận DTI <= 43%. Gọi sau verify_income."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "monthly_income": {"type": "number", "description": "Thu nhập tháng đã xác minh (triệu VNĐ)"},
                "existing_monthly_debt": {"type": "number", "description": "Tổng nợ hiện tại phải trả hàng tháng (triệu VNĐ)"},
                "requested_loan": {"type": "number", "description": "Số tiền vay yêu cầu (triệu VNĐ)"},
                "term_months": {"type": "integer", "description": "Số tháng vay"},
                "annual_rate": {
                    "type": "number",
                    "description": "Lãi suất năm (%)",
                    "default": 12.0
                }
            },
            "required": ["monthly_income", "existing_monthly_debt", "requested_loan", "term_months"]
        }
    },
    {
        "name": "generate_report",
        "description": (
            "Tạo báo cáo thẩm định và lưu vào CRM. Gọi SAU KHI đã hoàn thành "
            "tất cả các bước kiểm tra (credit, income, blacklist, DTI)."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "customer_id": {"type": "string"},
                "loan_amount": {"type": "number", "description": "Số tiền vay (triệu VNĐ)"},
                "decision": {
                    "type": "string",
                    "enum": ["APPROVED", "REJECTED", "NEEDS_HUMAN_REVIEW"],
                    "description": "Quyết định thẩm định"
                },
                "findings_summary": {"type": "string", "description": "Tóm tắt kết quả thẩm định"},
                "justification": {"type": "string", "description": "Lý do quyết định"}
            },
            "required": ["customer_id", "loan_amount", "decision", "findings_summary", "justification"]
        }
    }
]

print(f'✅ Đã định nghĩa {len(LOANBOT_TOOLS)} tools:')
for t in LOANBOT_TOOLS:
    required = t['input_schema']['required']
    print(f'  🔧 {t["name"]}  (required: {required})')

---

## 🎭 Bước 2: Xây Dựng Mock Tools (Giả Lập Hệ Thống Thực)

Trong prototype này, chúng ta **giả lập** các hệ thống backend (CIC, BHXH, NHNN).  
Trong Module 4, chúng ta sẽ thay thế bằng kết nối thực.

Chú ý cách mỗi mock function:
- Có **type hints** rõ ràng
- Trả về **dict có cấu trúc** (không phải string tự do)
- Mô phỏng **kết quả thực tế** dựa trên customer_id

In [ ]:
# ═══════════════════════════════════════════════════════
# BƯỚC 2: MOCK IMPLEMENTATIONS
# Giả lập kết nối với CIC, BHXH, NHNN, v.v.
# ═══════════════════════════════════════════════════════

# Dữ liệu khách hàng giả để test
MOCK_CUSTOMER_DB = {
    "FC-2024-001": {
        "name": "Nguyễn Văn An",
        "national_id": "001234567890",
        "credit_score": 720,
        "verified_income": 28.5,  # triệu VNĐ/tháng
        "existing_debt": 5.2,
        "is_blacklisted": False
    },
    "FC-2024-002": {
        "name": "Trần Thị Bình",
        "national_id": "009876543210",
        "credit_score": 580,
        "verified_income": 15.0,
        "existing_debt": 8.0,
        "is_blacklisted": False
    },
    "FC-2024-003": {
        "name": "Lê Văn Cường",
        "national_id": "031122334455",
        "credit_score": 650,
        "verified_income": 20.0,
        "existing_debt": 2.0,
        "is_blacklisted": True  # khách hàng nợ xấu!
    }
}


def mock_check_credit_score(customer_id: str, include_history: bool = True) -> dict:
    """Mock CIC credit score lookup."""
    customer = MOCK_CUSTOMER_DB.get(customer_id)
    if not customer:
        return {"error": f"Không tìm thấy khách hàng {customer_id}"}

    score = customer["credit_score"]
    if score >= 750:   risk = "rất thấp"
    elif score >= 700: risk = "thấp"
    elif score >= 650: risk = "trung bình"
    else:              risk = "cao"

    result = {
        "customer_id": customer_id,
        "customer_name": customer["name"],
        "credit_score": score,
        "risk_level": risk,
        "data_source": "CIC (Trung tâm Thông tin Tín dụng Quốc gia)",
        "retrieved_at": datetime.now().strftime("%Y-%m-%d %H:%M")
    }
    if include_history:
        result["payment_history"] = "Thanh toán đúng hạn 22/24 tháng gần nhất"
    return result


def mock_verify_income(customer_id: str, declared_income: float) -> dict:
    """Mock BHXH + bank statement income verification."""
    customer = MOCK_CUSTOMER_DB.get(customer_id)
    if not customer:
        return {"error": f"Không tìm thấy khách hàng {customer_id}"}

    verified = customer["verified_income"]
    diff_pct = abs(verified - declared_income) / declared_income * 100

    return {
        "customer_id": customer_id,
        "declared_income": declared_income,
        "verified_income": verified,
        "discrepancy_pct": round(diff_pct, 1),
        "income_sources": ["Lương công ty (BHXH)", "Thu nhập tự kinh doanh"],
        "confidence": "cao" if diff_pct < 10 else "trung bình",
        "data_source": "BHXH + Sao kê ngân hàng 6 tháng"
    }


def mock_check_blacklist(customer_id: str, national_id: str) -> dict:
    """Mock NHNN + internal blacklist check."""
    customer = MOCK_CUSTOMER_DB.get(customer_id)
    if not customer:
        return {"error": f"Không tìm thấy khách hàng {customer_id}"}

    blacklisted = customer["is_blacklisted"]
    return {
        "customer_id": customer_id,
        "national_id": national_id,
        "is_blacklisted": blacklisted,
        "reason": "Nợ xấu nhóm 5 tại ACB (2023)" if blacklisted else None,
        "severity": "CRITICAL" if blacklisted else None,
        "checked_sources": ["NHNN Blacklist", "FinTech Corp Internal", "CIC Risk"]
    }


def mock_calculate_dti(
    monthly_income: float,
    existing_monthly_debt: float,
    requested_loan: float,
    term_months: int,
    annual_rate: float = 12.0
) -> dict:
    """Calculate DTI ratio accurately."""
    # Monthly payment using annuity formula
    monthly_rate = annual_rate / 100 / 12
    if monthly_rate > 0:
        monthly_payment = requested_loan * (monthly_rate * (1 + monthly_rate)**term_months) / \
                          ((1 + monthly_rate)**term_months - 1)
    else:
        monthly_payment = requested_loan / term_months

    total_debt = existing_monthly_debt + monthly_payment
    dti_ratio = total_debt / monthly_income * 100

    return {
        "monthly_income": monthly_income,
        "new_loan_monthly_payment": round(monthly_payment, 2),
        "total_monthly_debt": round(total_debt, 2),
        "dti_ratio_pct": round(dti_ratio, 1),
        "max_allowed_dti": 43.0,
        "passes_dti_check": dti_ratio <= 43.0,
        "affordability": "Tốt" if dti_ratio <= 35 else ("Chấp nhận được" if dti_ratio <= 43 else "Quá cao")
    }


def mock_generate_report(
    customer_id: str,
    loan_amount: float,
    decision: str,
    findings_summary: str,
    justification: str
) -> dict:
    """Generate and save assessment report."""
    report_id = f"RPT-{datetime.now().strftime('%Y%m%d%H%M%S')}-{customer_id[-3:]}"
    return {
        "report_id": report_id,
        "customer_id": customer_id,
        "loan_amount": loan_amount,
        "decision": decision,
        "generated_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "status": "saved_to_crm",
        "report_url": f"https://crm.fintechcorp.vn/reports/{report_id}",
        "findings_summary": findings_summary
    }


# Map tool name → function
TOOL_FUNCTIONS = {
    "check_credit_score": mock_check_credit_score,
    "verify_income":      mock_verify_income,
    "check_blacklist":    mock_check_blacklist,
    "calculate_dti":      mock_calculate_dti,
    "generate_report":    mock_generate_report,
}

print('✅ Tất cả mock tools đã sẵn sàng!')
print(f'   Có {len(MOCK_CUSTOMER_DB)} khách hàng trong mock database')

# Test nhanh một tool
test_result = mock_check_credit_score('FC-2024-001')
print(f'\n🧪 Test check_credit_score (FC-2024-001):')
for k, v in test_result.items():
    print(f'   {k}: {v}')

---

## ⚡ Bước 3: Tool Dispatcher — Trái Tim Của Harness

**Tool Dispatcher** là component trong harness nhận yêu cầu gọi tool từ LLM và:
1. Tìm function tương ứng
2. Validate tham số đầu vào
3. Thực thi function
4. Trả kết quả về cho LLM

Đây là **phần nhỏ nhất** của Tool Orchestration layer — trong Module 2, chúng ta sẽ thêm retry logic, error handling, và permission checking.

In [ ]:
# ═══════════════════════════════════════════════════════
# BƯỚC 3: TOOL DISPATCHER (Minimal Harness Component)
# ═══════════════════════════════════════════════════════

def dispatch_tool(tool_name: str, tool_input: dict) -> str:
    """
    Nhận yêu cầu gọi tool từ LLM, thực thi, trả kết quả dạng JSON string.
    Đây là 'minimal Tool Orchestration layer' — phiên bản đơn giản nhất.
    """
    print(f'  🔧 Dispatching: {tool_name}({json.dumps(tool_input, ensure_ascii=False)})')

    fn = TOOL_FUNCTIONS.get(tool_name)
    if fn is None:
        result = {"error": f"Tool '{tool_name}' không tồn tại"}
    else:
        try:
            result = fn(**tool_input)
        except Exception as e:
            result = {"error": f"Lỗi thực thi tool: {str(e)}"}

    print(f'  ✅ Kết quả: {json.dumps(result, ensure_ascii=False)[:120]}...' if len(str(result)) > 120 else f'  ✅ Kết quả: {result}')
    return json.dumps(result, ensure_ascii=False)


# Test dispatcher
print('🧪 Test Tool Dispatcher:')
test = dispatch_tool('calculate_dti', {
    'monthly_income': 28.5,
    'existing_monthly_debt': 5.2,
    'requested_loan': 200,
    'term_months': 48
})
print(f'\n📋 Full result: {json.loads(test)}')

---

## 🤖 Bước 4: Agent Loop — Vòng Lặp Agentic

Đây là **trái tim của AI Agent**: vòng lặp nhận tin nhắn, gọi Claude API,  
xử lý tool calls, và tiếp tục cho đến khi agent hoàn thành nhiệm vụ.

Trong bài giảng chúng ta đã học Agentic Loop:
```
Perceive → Think → Act (tool call) → Observe (result) → lặp lại hoặc kết thúc
```

Code bên dưới implement chính xác vòng lặp đó:

In [ ]:
# ═══════════════════════════════════════════════════════
# BƯỚC 4: AGENT LOOP — Minimal Agentic Harness
# ═══════════════════════════════════════════════════════

SYSTEM_PROMPT = """Bạn là LoanBot — AI agent thẩm định hồ sơ vay của FinTech Corp.

TRÁCH NHIỆM:
Khi nhận hồ sơ vay, bạn PHẢI thực hiện ĐẦY ĐỦ các bước sau theo thứ tự:
1. check_credit_score — Kiểm tra điểm tín dụng
2. check_blacklist — Kiểm tra danh sách đen (nếu blacklisted → REJECTED ngay)
3. verify_income — Xác minh thu nhập
4. calculate_dti — Tính DTI ratio (nếu DTI > 43% → REJECTED)
5. generate_report — Tạo báo cáo và lưu kết quả

TIÊU CHÍ PHÊ DUYỆT:
- Credit score >= 650
- Không trong danh sách đen
- Thu nhập xác minh đạt yêu cầu
- DTI ratio <= 43%
- Khoản vay > 500 triệu → chuyển NEEDS_HUMAN_REVIEW

Luôn dùng tools để lấy dữ liệu thực. KHÔNG được bịa đặt bất kỳ số liệu nào.
Báo cáo cuối cùng bằng tiếng Việt, rõ ràng, chuyên nghiệp."""


def run_loanbot(loan_request: str, max_iterations: int = 10) -> str:
    """
    Chạy LoanBot Agent Loop cho một hồ sơ vay.
    
    Args:
        loan_request: Mô tả hồ sơ vay bằng ngôn ngữ tự nhiên
        max_iterations: Giới hạn vòng lặp (guardrail chống vòng lặp vô hạn)
    
    Returns:
        Kết quả thẩm định cuối cùng
    """
    print('=' * 60)
    print('🤖 LOANBOT BẮT ĐẦU XỬ LÝ HỒ SƠ')
    print('=' * 60)
    print(f'📋 Yêu cầu: {loan_request[:100]}...' if len(loan_request) > 100 else f'📋 Yêu cầu: {loan_request}')
    print()

    messages = [{"role": "user", "content": loan_request}]
    iteration = 0

    while iteration < max_iterations:
        iteration += 1
        print(f'--- Vòng lặp {iteration} ---')

        # ── THINK: Gọi Claude để quyết định bước tiếp theo ──
        response = client.messages.create(
            model="claude-haiku-4-5-20251001",  # Dùng Haiku để tiết kiệm chi phí cho lab
            max_tokens=1024,
            system=SYSTEM_PROMPT,
            tools=LOANBOT_TOOLS,
            messages=messages
        )

        # ── OBSERVE: Kiểm tra response ──
        print(f'  📊 Stop reason: {response.stop_reason}')

        # Thêm response của assistant vào messages
        messages.append({"role": "assistant", "content": response.content})

        # Nếu không có tool call → agent đã hoàn thành
        if response.stop_reason == "end_turn":
            final_text = ""
            for block in response.content:
                if hasattr(block, 'text'):
                    final_text = block.text
            print()
            print('=' * 60)
            print('✅ LOANBOT HOÀN THÀNH')
            print('=' * 60)
            return final_text

        # ── ACT: Thực thi tất cả tool calls trong response này ──
        if response.stop_reason == "tool_use":
            tool_results = []

            for block in response.content:
                if block.type == "tool_use":
                    # Dispatch tool call
                    result_str = dispatch_tool(block.name, block.input)
                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": result_str
                    })

            # Đưa kết quả tool vào messages để agent tiếp tục
            messages.append({"role": "user", "content": tool_results})
            print()

    return "⚠️ Vượt quá giới hạn vòng lặp. Cần Human Review."


print('✅ Agent Loop đã sẵn sàng!')

---

## 🧪 Bước 5: Test Case 1 — Hồ Sơ Bình Thường (Nên Duyệt)

**Khách hàng:** Nguyễn Văn An (FC-2024-001)  
- Credit score: 720 (tốt)
- Thu nhập xác minh: 28.5 triệu/tháng
- Không trong blacklist
- Yêu cầu: Vay 200 triệu, 48 tháng, mua xe

**Kỳ vọng:** LoanBot duyệt hồ sơ sau khi kiểm tra đủ các bước

In [ ]:
# Test Case 1: Hồ sơ tốt
result1 = run_loanbot("""
Hồ sơ vay mới cần thẩm định:
- Mã khách hàng: FC-2024-001
- CCCD: 001234567890
- Họ tên: Nguyễn Văn An
- Thu nhập khai báo: 30 triệu VNĐ/tháng
- Số tiền vay: 200 triệu VNĐ
- Kỳ hạn: 48 tháng
- Mục đích: Mua xe ô tô
- Lãi suất áp dụng: 12%/năm

Hãy thẩm định đầy đủ và tạo báo cáo kết quả.
""")

print("\n📄 KẾT QUẢ CUỐI CÙNG:")
print(result1)

---

## 🧪 Bước 6: Test Case 2 — Khách Hàng Trong Blacklist (Nên Từ Chối)

**Khách hàng:** Lê Văn Cường (FC-2024-003)  
- Trong danh sách nợ xấu NHNN!

**Kỳ vọng:** LoanBot từ chối ngay sau khi phát hiện blacklist, không cần kiểm tra thêm

In [ ]:
# Test Case 2: Khách hàng blacklisted
result2 = run_loanbot("""
Hồ sơ vay mới cần thẩm định:
- Mã khách hàng: FC-2024-003
- CCCD: 031122334455
- Họ tên: Lê Văn Cường
- Thu nhập khai báo: 25 triệu VNĐ/tháng
- Số tiền vay: 150 triệu VNĐ
- Kỳ hạn: 36 tháng
- Mục đích: Vay kinh doanh
""")

print("\n📄 KẾT QUẢ CUỐI CÙNG:")
print(result2)

---

## 💡 Bước 7: Phân Tích và Reflection

Sau khi chạy hai test case, hãy quan sát:

In [ ]:
print("""📊 PHÂN TÍCH: Điều gì đã xảy ra trong Agent Loop?

=== TEST CASE 1 (Nguyễn Văn An — Duyệt) ===
Agentic Loop hoạt động:
  Vòng 1: Perceive (nhận hồ sơ) → Think (quyết định gọi check_credit_score)
            → Act (gọi tool) → Observe (score=720, risk=low)
  Vòng 2: Think (quyết định check_blacklist) → Act → Observe (not blacklisted)
  Vòng 3: Think (verify_income) → Act → Observe (28.5tr/tháng confirmed)
  Vòng 4: Think (calculate_dti) → Act → Observe (DTI=x%, passes check)
  Vòng 5: Think (generate_report, APPROVED) → Act → Observe (report saved)
  Vòng 6: end_turn → HOÀN THÀNH

=== TEST CASE 2 (Lê Văn Cường — Từ chối) ===
  Vòng 1: check_credit_score → score=650
  Vòng 2: check_blacklist → is_blacklisted=TRUE → DỪNG!
  Vòng 3: generate_report (REJECTED) → báo cáo
  Vòng 4: end_turn → HOÀN THÀNH (nhanh hơn!)

=== LIÊN HỆ VỚI BÀI GIẢNG ===
✅ Phương trình Agent = Model + Harness đã được implement:
   - Model: Claude Haiku (suy luận, quyết định tool call)
   - Harness: LOANBOT_TOOLS + dispatch_tool() + Agent Loop

✅ Hallucination đã được ngăn chặn:
   - LoanBot KHÔNG được bịa số liệu (system prompt)
   - Mọi số liệu phải đến từ tool calls

⚠️ Những gì còn thiếu (sẽ học Module 2-6):
   - Memory: nhớ hồ sơ qua các session
   - Verification Loop: validate kết quả tool trước khi tiếp tục
   - Retry logic: nếu tool fail, thử lại
   - Observability: log mọi bước để audit
   - Cost tracking: đếm token đã dùng
""")

# Đếm token thực tế
print("\nChú ý: Với API thực, mỗi tool call tốn tokens.")
print("Trong production, cần Budget Tracker để kiểm soát chi phí (Module 4).")

---

## ✏️ Bài Tập Tự Làm

### Bài tập 1 (Cơ bản)
Chạy Test Case 3 với **Trần Thị Bình (FC-2024-002)**:  
- Vay 300 triệu, 60 tháng, mua nhà  
- CCCD: 009876543210  
- Thu nhập khai báo: 18 triệu/tháng  

**Câu hỏi:** LoanBot sẽ quyết định gì? Tại sao?

### Bài tập 2 (Nâng cao)
Thêm một tool mới: `check_employment_status` với input `customer_id` và output `employer`, `employment_type` (permanent/contract), `tenure_months`.  
Cập nhật system prompt để LoanBot gọi tool này sau verify_income.

### Bài tập 3 (Suy nghĩ)
Nếu mock_check_credit_score trả về `{"error": "CIC system unavailable"}`,  
LoanBot hiện tại sẽ xử lý thế nào? Điều gì có thể sai?  
Chúng ta cần thêm gì vào harness để xử lý trường hợp này?

In [ ]:
# Bài tập 1: Điền vào chỗ trống và chạy
result3 = run_loanbot("""
Hồ sơ vay mới cần thẩm định:
- Mã khách hàng: FC-2024-002
- CCCD: 009876543210
- Họ tên: Trần Thị Bình
- Thu nhập khai báo: 18 triệu VNĐ/tháng
- Số tiền vay: 300 triệu VNĐ
- Kỳ hạn: 60 tháng
- Mục đích: Mua nhà
- Lãi suất: 11%/năm
""")

print("\n📄 KẾT QUẢ:")
print(result3)

In [ ]:
# Bài tập 2: Thêm tool check_employment_status
# TODO: Thêm định nghĩa tool vào LOANBOT_TOOLS
# TODO: Thêm mock function mock_check_employment_status
# TODO: Thêm vào TOOL_FUNCTIONS
# TODO: Cập nhật SYSTEM_PROMPT

print("💡 Gợi ý cấu trúc mock function:")
print("""
def mock_check_employment_status(customer_id: str) -> dict:
    # Trả về employer, employment_type, tenure_months
    ...
""")

---

## 🎓 Tổng Kết Lab Module 1

Trong lab này bạn đã:

| Thành tựu | Liên quan đến bài giảng |
|-----------|------------------------|
| ✅ Định nghĩa 5 tool schemas | Bài 4: Tool Definition |
| ✅ Xây dựng mock tool implementations | Bài 4: Function Calling |
| ✅ Implement Tool Dispatcher | Bài 3: Harness layer 1 |
| ✅ Viết Agent Loop (Agentic Loop) | Bài 2: Perceive→Think→Act→Observe |
| ✅ Test với 3 test cases khác nhau | Thực tế doanh nghiệp |
| ✅ Quan sát LLM tự quyết định tool call | Bài 3: Model + Harness |

**Module tiếp theo — Module 2: Kiến Trúc Harness**  
Chúng ta sẽ mở rộng harness này thành 5-layer production architecture:  
Tool Orchestration + Verification Loops + Context & Memory + Guardrails + Observability

---
*AI Agent Harness Training · Module 1 Lab · FinTech Corp / LoanBot*